# Offline with Parquet

The Parquet backend reads seqout data with DuckDB and does not call the API.
Use it for offline work, large batch jobs, and your own SQL.

In [1]:
from seqoutdb import connect_to_seqout

pq = connect_to_seqout(backend="parquet")

## Set the data source

The source is a URL or a local directory. This example uses the public URL,
`https://seqout.org/data`. You can also download the files from that same
address and point the source at a local directory (see the last cell).

In [2]:
pq.set_source("https://seqout.org/data")

## Get a study

The Parquet backend has the same methods as the API backend.

In [3]:
study = pq.fetch_study("GSE169470")
print(study.title)
print(study.num_experiments, "experiments,", study.num_samples, "samples")

RNA-sequencing Analysis (RNA-seq)，Genome-wide Maps of Chromatin State (ChIP-seq) and Assay for Transposase Accessible Chromatin with High-throughput Sequencing (ATAC-seq) in BMDMs or Raw 264.7 cell lines.
47 experiments, 47 samples


## Run your own SQL

`execute_query` runs SQL with DuckDB. The backend maps each table name to its
Parquet file. Do not reuse a table name as a column alias, because the backend
finds table names by text; use a distinct alias such as `AS n`. The result
converts to pandas with `.df()`.

In [4]:
pq.execute_query(
    "SELECT source, COUNT(*) AS n FROM unified_metadata GROUP BY source ORDER BY n DESC"
).df()

,source,n
0,sra,684870
1,geo,273727
2,ena,243708
3,ae,21008


## Work with a local copy

A query over a remote URL is slow for the large tables. Download the files
once, then query them from disk:

```bash
seqoutdb parquet download /data/seqout
```

In [5]:
# pq.set_source("/data/seqout")   # after you download the files
# print(len(pq.fetch_study_runs("SRP311850")), "runs")